In [ ]:
import sys
sys.path.insert(0, "src")  # /workspace/src

from patent_train import TrainingRunner, TrainConfig

In [ ]:
SEARCH = False   # True=fast-fail 짧은 런(2 epoch로 유도) / False=풀런(아래 epochs 사용)

cfg = TrainConfig(
    backbone="axenc_tapt",   # TAPT 위해 백본 변경
    loss="focal",
    loss_params={"alpha": 0.25, "gamma": 2},
    max_len=512,
    eff_batch=128,          # 배치 재현 파라미터
    micro_batch=128,        
    eval_micro_batch=512,
    learning_rate=4.8e-4,   # 확정 레시피 lr
    weight_decay=0.01,
    warmup_ratio=0.1,
    epochs=12,
    early_stop_epochs=2,    # 개선 없이 견디는 에폭 수(eval 횟수 환산은 runner가 처리)
    notebook_name="13_02_TAPT_Train.ipynb",   # wandb code saving
    tag="modernbert-patent-len512-tapt",
    run_name="axenc_len512_tapt_train",
    repo_final="ingyoun/A.X-patent-len512-tapt",
    out_path="/workspace/output/modernbert-len512-tapt",
    search=SEARCH,
)

print("run_name:", cfg.run_name, "| epochs:", cfg.epochs, "| grad_accum:", cfg.grad_accum)

## 구성 — 데이터·모델

토크나이저+원본 로드 → `max_len` 절단(캐시) → 분류기 구성. 단계를 나눠 중간 점검·부분 재실행이 가능하다.

In [ ]:
runner = TrainingRunner(cfg)

In [ ]:
runner.load_data()        # 토크나이저 + 원본 데이터셋(prep 캐시 있으면 원본 생략)
runner.data.raw

In [ ]:
runner.prepare_data()     # max_len 절단 → prep 캐시
runner.data.dataset

In [ ]:
runner.load_model()

## 훈련

In [ ]:
runner.build_trainer()

In [ ]:
runner.train()

## 평가 · 메트릭 저장 · push

모델 가중치는 로컬에 두지 않고 Hub로만 올린다(팟을 지우면 로컬 사본은 사라진다). 로컬 사본이 필요하면 `runner.save_model()`.

In [ ]:
test_metrics = runner.evaluate("test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

In [ ]:
runner.save_metrics()     # runner.metrics(split 전체) → {tag}_metrics.json

In [ ]:
runner.push_to_hub()

## val·test 로짓 덤프

`logits_{tag}_{split}.npy`를 `out_path` 상위(`/workspace/output/`)에 저장

In [ ]:
runner.predict_logits("val")
runner.predict_logits("test")